In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import cosine
from collections import defaultdict

PROCESSED = Path("../Dataset/Processed")
df_meta = pd.read_csv("../Dataset/Raw/games_detailed_info.csv", low_memory=False).rename(columns={"primary":"name"}).set_index("id")
desc_sim = joblib.load(PROCESSED / "desc_topk_csr.joblib")
content_sim = joblib.load(PROCESSED / "content_topk_csr.joblib")
emb_sim = joblib.load(PROCESSED / "emb_topk_csr.joblib")
cf_algo = None

MemoryError: Unable to allocate 3.49 GiB for an array with shape (467900161,) and data type float64

In [ ]:
meta_ids = df_meta.index.to_numpy()
id_to_idx = {int(gid): idx for idx, gid in enumerate(meta_ids)}
idx_to_id = {v:k for k,v in id_to_idx.items()}

def topk_from_sim(seed_ids, sim_csr, k=500, weight_strategy="mean"):
    """
    seed_ids: list of game IDs (BGG ids)
    sim_csr: CSR sparse similarity [N, N]
    Return: pd.Series(index=game_id, values=score) for candidate neighbors only.
    """
    seed_idxs = [id_to_idx[sid] for sid in seed_ids if sid in id_to_idx]
    if not seed_idxs:
        return pd.Series(dtype=float)

    scores = defaultdict(float)
    counts = defaultdict(int)

    for si in seed_idxs:
        row = sim_csr.getrow(si)
        cols = row.indices
        vals = row.data

        for c, v in zip(cols, vals):
            gid = idx_to_id[c]
            if weight_strategy == "max":
                scores[gid] = max(scores[gid], float(v))
            else:
                scores[gid] += float(v)
                counts[gid] += 1

    if weight_strategy != "max":
        for gid in list(scores.keys()):
            scores[gid] = scores[gid] / max(counts[gid], 1)

    # remove seeds
    for sid in seed_ids:
        scores.pop(sid, None)

    # take top-k
    items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return pd.Series(dict(items), dtype=float)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

def hybrid_recommend(seed_game_ids, alpha=0.4, beta=0.25, gamma=0.25, delta=0.1, topk=20):
    """
    alpha = weight CF
    beta = desc/content similarity
    gamma = embedding similarity
    delta = sentiment boost
    returns DataFrame of recommendations with combined score
    """
    cf_series = cf_scores_for_seed(seed_game_ids)
    cf_series = cf_series.reindex(meta_ids).fillna(cf_series.mean())
    s = MinMaxScaler()
    cf_norm = pd.Series(s.fit_transform(cf_series.values.reshape(-1,1)).flatten(), index=meta_ids)
    
    CANDIDATE_K = 800

    desc_series = topk_from_sim(seed_game_ids, desc_sim, k=CANDIDATE_K).reindex(meta_ids).fillna(0)
    content_series = topk_from_sim(seed_game_ids, content_sim, k=CANDIDATE_K).reindex(meta_ids).fillna(0)
    desc_norm = pd.Series(MinMaxScaler().fit_transform(desc_series.values.reshape(-1,1)).flatten(), index=meta_ids)
    content_norm = pd.Series(MinMaxScaler().fit_transform(content_series.values.reshape(-1,1)).flatten(), index=meta_ids)

    emb_series = topk_from_sim(seed_game_ids, emb_sim, k=CANDIDATE_K).reindex(meta_ids).fillna(0)
    emb_norm = pd.Series(MinMaxScaler().fit_transform(emb_series.values.reshape(-1,1)).flatten(), index=meta_ids)

    sent_summary = pd.read_csv(PROCESSED / "sentiment_summary.csv", index_col="id") if (PROCESSED/"sentiment_summary.csv").exists() else None
    if sent_summary is not None:
        sent = sent_summary["avg_sentiment_score"].reindex(meta_ids).fillna(sent_summary["avg_sentiment_score"].mean())
        sent_norm = pd.Series(MinMaxScaler().fit_transform(sent.values.reshape(-1,1)).flatten(), index=meta_ids)
    else:
        sent_norm = pd.Series(0.0, index=meta_ids)

    text_norm = 0.5*desc_norm + 0.5*content_norm
    combined = alpha*cf_norm + beta*text_norm + gamma*emb_norm + delta*sent_norm
    
    for sid in seed_game_ids:
        if sid in combined.index:
            combined.loc[sid] = -1
    recs = combined.sort_values(ascending=False).head(topk)
    df_out = pd.DataFrame({
        "game_id": recs.index,
        "score": recs.values,
        "name": [df_meta.loc[g,"name"] if g in df_meta.index else "" for g in recs.index]
    })
    return df_out

In [ ]:
if not (PROCESSED/"sentiment_summary.csv").exists():
    df_rev = pd.read_csv(PROCESSED/"reviews_with_sentiment.csv", low_memory=False)
    sent_summary = (
        df_rev.groupby("ID")
        .agg(avg_sentiment_score=("sentiment_score","mean"), review_count=("ID","count"))
        .reset_index().rename(columns={"ID":"id"}).set_index("id")
    )
    sent_summary.to_csv(PROCESSED/"sentiment_summary.csv")
    print("saved sentiment_summary.csv")
else:
    print("sentiment_summary exists.")

In [ ]:
seed_ids = [13]
recs = hybrid_recommend(seed_ids, alpha=0.4, beta=0.3, gamma=0.2, delta=0.1, topk=15)
display(recs.head(15))